# Checks

## Train `history.csv`

### Table length

Total number of rows should be 5912
- Currently 5634. ❌ _ingestion might have some issues as the rows are not matching_. 

In [0]:
SELECT COUNT(1)
FROM workspace.default.train

In [0]:
SELECT *
FROM workspace.default.train
LIMIT 10

### Variables: 

- Expected: 23 columns
    - Reality: 23 columns. ✅

| #   | Variable         | Type        |
| --- | ---------------- | ----------- |
| 1   | customerID       | ID          |
| 2   | gender           | Categorical |
| 3   | SeniorCitizen    | Categorical |
| 4   | Partner          | Categorical |
| 5   | Dependents       | Categorical |
| 6   | tenure           | Categorical |
| 7   | PhoneService     | Categorical |
| 8   | MultipleLines    | Categorical |
| 9   | InternetService  | Categorical |
| 10  | OnlineSecurity   | Categorical |
| 11  | OnlineBackup     | Categorical |
| 12  | DeviceProtection | Categorical |
| 13  | TechSupport      | Categorical |
| 14  | StreamingTV      | Categorical |
| 15  | StreamingMovies  | Categorical |
| 16  | Contract         | Categorical |
| 17  | PaperlessBilling | Categorical |
| 18  | PaymentMethod    | Categorical |
| 19  | MonthlyCharges   | Numerical   |
| 20  | TotalCharges     | Numerical   |
| 21  | Churn            | Categorical |
| 22  | CustomerFeedback | Text        |
| 23  | MonthlyIncome    | Numerical   |




In [0]:
SELECT
    tenure,
    COUNT(1) AS qty
FROM workspace.default.train
GROUP BY ALL
ORDER BY qty


In [0]:
DESCRIBE workspace.default.train;
-- DESCRIBE EXTENDED workspace.default.train;

## Inference `inference.csv`

### Table length

Total number of rows should be 1461
- Currently 1409. ❌ _ingestion might have some issues as the rows are not matching_. 



In [0]:
SELECT COUNT(1)
FROM workspace.default.inference

In [0]:
SELECT *
FROM workspace.default.inference
LIMIT 10

### Variables

Inference table has 22 variables, they all match with Train except for the Chrun variable whihc makes sense and it is expected.

In [0]:
DESCRIBE workspace.default.inference;
-- DESCRIBE EXTENDED workspace.default.inference;

## train vs inference: CustomerID

Do we have duplicated customers?
- It dos not look like.
- Each customer is unique in both tables.



In [0]:
SELECT
    CustomerID,
    MAX(in_train)     AS in_train,
    MAX(in_inference) AS in_inference
FROM (
    SELECT CustomerID, 1 AS in_train, 0 AS in_inference FROM workspace.default.train
    UNION ALL
    SELECT CustomerID, 0 AS in_train, 1 AS in_inference FROM workspace.default.inference
) AS t
GROUP BY CustomerID;

# Feature Engineering

Initiating the cleaning process that will be used to create the feature store.

In [0]:

DROP TABLE IF EXISTS workspace.default.train_silver;
CREATE TABLE IF NOT EXISTS workspace.default.train_silver AS
WITH 
main_tb AS (
    SELECT *
    FROM workspace.default.train
),

standardize_variables AS (
    SELECT 
        customerID,

        -- IDENTITY
        -- gender, 
        (CASE WHEN lower(gender)="male" THEN 1 ELSE 0 END) AS isMale,
        SeniorCitizen,
        -- Partner,
        (CASE WHEN lower(Partner)="yes" THEN 1 ELSE 0 END) AS isPartner,
        -- Dependents,
        (CASE WHEN lower(Dependents)="yes" THEN 1 ELSE 0 END) AS HasDependents,
        MonthlyIncome,

        -- ACCOUNT
        -- tenure,
        TRY_CAST(tenure AS BIGINT) AS nbrTenure,
        -- Contract,
        (CASE 
            WHEN lower(Contract)="month-to-month" THEN "M2M"
            WHEN lower(Contract)="one year" THEN "one-year" 
            WHEN lower(Contract)="two year" THEN "two-year" 
            ELSE NULL
        END) AS ContractTier,
        (CASE WHEN lower(Contract)="one year" OR lower(Contract)="two year" THEN 1 ELSE 0 END) AS isContractLong,
        (CASE WHEN lower(Contract)="one year" THEN 1 ELSE 0 END) AS isContractOneYear,
        (CASE WHEN lower(Contract)="two year" THEN 1 ELSE 0 END) AS isContractTwoYear,
        -- PaperlessBilling,
        (CASE WHEN lower(PaperlessBilling)="yes" THEN 1 ELSE 0 END) AS isPaperlessBilling,
        -- PaymentMethod,
        (CASE WHEN lower(PaymentMethod) LIKE "%automatic%" THEN 1 ELSE 0 END) AS isPaymentMethodAuto,
        (CASE WHEN lower(PaymentMethod) LIKE "%check%" THEN 1 ELSE 0 END) AS isPaymentMethodCheck,
        (CASE WHEN lower(PaymentMethod) LIKE "bank transfer%" THEN 1 ELSE 0 END) AS isPaymentMethod_BankTransfer,
        (CASE WHEN lower(PaymentMethod) LIKE "credit card%" THEN 1 ELSE 0 END) AS isPaymentMethod_CreditCard,
        (CASE WHEN lower(PaymentMethod) LIKE "electronic check%" THEN 1 ELSE 0 END) AS isPaymentMethod_ElectronicCheck,
        (CASE WHEN lower(PaymentMethod) LIKE "mailed check%" THEN 1 ELSE 0 END) AS isPaymentMethod_MailedCheck,

        -- PHONE SERVICES
        -- PhoneService,
        (CASE WHEN lower(PhoneService)="yes" THEN 1 ELSE 0 END) AS hasPhoneService,
        -- MultipleLines,
        (CASE 
            WHEN lower(MultipleLines)="yes" THEN 1 
            WHEN lower(MultipleLines)="no" THEN 0 
            WHEN lower(MultipleLines)="no phone service" THEN 0 
            ELSE NULL 
        END) AS hasMultipleLines,

        -- INTERNET SERVICES
        -- InternetService,
        (CASE WHEN lower(InternetService)="dsl" OR lower(InternetService)="fiber optic" THEN 1 ELSE 0 END) AS hasInternetService,
        (CASE WHEN lower(InternetService)="dsl" THEN 1 ELSE 0 END) AS isInternetService_DSL,
        (CASE WHEN lower(InternetService)="fiber optic" THEN 1 ELSE 0 END) AS isInternetService_Fiber,

        -- INTERNET SERVICES ADD-ON
        -- OnlineSecurity,
        (CASE WHEN lower(OnlineSecurity)="yes" THEN 1 ELSE 0 END) AS hasOnlineSecurity,
        -- OnlineBackup,
        (CASE WHEN lower(OnlineBackup)="yes" THEN 1 ELSE 0 END) AS hasOnlineBackup,
        -- DeviceProtection,
        (CASE WHEN lower(DeviceProtection)="yes" THEN 1 ELSE 0 END) AS hasDeviceProtection,
        -- TechSupport,
        (CASE WHEN lower(TechSupport)="yes" THEN 1 ELSE 0 END) AS hasTechSupport,
        -- StreamingTV,
        (CASE WHEN lower(StreamingTV)="yes" THEN 1 ELSE 0 END) AS hasStreamingTV,
        -- StreamingMovies,
        (CASE WHEN lower(StreamingMovies)="yes" THEN 1 ELSE 0 END) AS hasStreamingMovies,

        -- BILLING
        MonthlyCharges,
        TotalCharges,
        (CASE 
            WHEN MonthlyCharges < 30 THEN '0-30'
            WHEN MonthlyCharges < 70 THEN '30-70'
            WHEN MonthlyCharges < 90 THEN '70-90'
            ELSE '90+'
        END) AS ChargeBand,

        -- TARGET
        -- Churn,
        (CASE WHEN lower(Churn) = "yes" THEN 1 ELSE 0 END) AS TargetChurn,

        -- FEEDBACK
        -- CustomerFeedback,
        TRIM(BOTH '"' FROM CustomerFeedback) AS CustomerFeedback_adj

    FROM main_tb
) ,

-- Add derived variables
derived_variables AS (
    SELECT 
        *,
        -- ACCOUNT
        (CASE WHEN nbrTenure <= 6 THEN 1 ELSE 0 END) AS isNewCustomer,
        (CASE 
            WHEN nbrTenure <=  6 THEN "1_0to6"
            WHEN nbrTenure <= 12 THEN "2_7to12"
            WHEN nbrTenure <= 24 THEN "3_13to24"
            WHEN nbrTenure <= 48 THEN "4_25to48"
            WHEN nbrTenure >  48 THEN "5_49plus"
            ELSE "6_other"
        END) AS tenureGroup,

        -- INTERNET SERVICES ADDON
        (hasOnlineSecurity + hasOnlineBackup + hasDeviceProtection + hasTechSupport + hasStreamingTV + hasStreamingMovies) AS nbrInternetServicesAddOnQty,
        (hasOnlineSecurity + hasOnlineBackup + hasDeviceProtection + hasTechSupport + hasStreamingTV + hasStreamingMovies) / 6.0 AS nbrInternetServicesAddOnQtyPct,
        (CASE WHEN hasOnlineSecurity + hasOnlineBackup + hasDeviceProtection = 3 THEN 1 ELSE 0 END) AS hasSecurityBundle,
        (CASE WHEN  hasStreamingTV + hasStreamingMovies > 1 THEN 1 ELSE 0 END) AS hasStreamingBundle,
        (hasStreamingTV + hasStreamingMovies) AS nbrStreamingCount,

        -- Service Tier
        (CASE 
            WHEN hasPhoneService = 1 AND hasInternetService = 1 THEN "dual"
            WHEN hasPhoneService = 1 AND hasInternetService = 0 THEN "phone_only"
            WHEN hasPhoneService = 0 AND hasInternetService = 1 THEN "internet_only"
            ELSE NULL 
        END) AS ServiceTier,

        -- CROSS VARIABLES
        -- Total Spend
        (nbrTenure * MonthlyCharges) AS TotalSpend,
        TRY_DIVIDE(TotalCharges, TotalSpend) AS TotalSpendPct,

        -- Contract Tier
        (CASE WHEN ContractTier="M2M" AND (hasPhoneService = 1 AND hasInternetService = 1) THEN 1 ELSE 0 END) AS isM2M_dual,
        (CASE WHEN ContractTier="M2M" AND (hasPhoneService = 0 AND hasInternetService = 1) THEN 1 ELSE 0 END) AS isM2M_internetOnly,
        (CASE WHEN ContractTier="one-year" AND (hasPhoneService = 1 AND hasInternetService = 1) THEN 1 ELSE 0 END) AS isOneYear_dual,
        (CASE WHEN ContractTier="two-year" AND (hasPhoneService = 1 AND hasInternetService = 1) THEN 1 ELSE 0 END) AS isTwoYear_dual,

        -- Calcualted
        ROUND(TRY_DIVIDE(TotalCharges, MonthlyCharges),0) AS tenure_calc,
        TRY_DIVIDE(MonthlyCharges, MonthlyIncome) AS chargeRatio,
        (hasPhoneService + hasInternetService + hasMultipleLines + nbrInternetServicesAddOnQty) AS numServices,
         TRY_DIVIDE(MonthlyCharges, hasPhoneService + hasInternetService + hasMultipleLines + nbrInternetServicesAddOnQty) AS chargesPerService

    FROM standardize_variables
)

SELECT
    customerID,
    isMale,
    SeniorCitizen,
    isPartner,
    HasDependents,
    MonthlyIncome,
    (CASE WHEN nbrTenure IS NULL THEN tenure_calc ELSE nbrTenure END) AS nbrTenure,
    ContractTier,
    isContractLong,
    isContractOneYear,
    isContractTwoYear,
    isPaperlessBilling,
    isPaymentMethodAuto,
    isPaymentMethodCheck,
    isPaymentMethod_BankTransfer,
    isPaymentMethod_CreditCard,
    isPaymentMethod_ElectronicCheck,
    isPaymentMethod_MailedCheck,
    hasPhoneService,
    hasMultipleLines,
    hasInternetService,
    isInternetService_DSL,
    isInternetService_Fiber,
    hasOnlineSecurity,
    hasOnlineBackup,
    hasDeviceProtection,
    hasTechSupport,
    hasStreamingTV,
    hasStreamingMovies,
    MonthlyCharges,
    TotalCharges,
    isNewCustomer,
    tenureGroup,
    nbrInternetServicesAddOnQty,
    nbrInternetServicesAddOnQtyPct,
    hasSecurityBundle,
    hasStreamingBundle,
    nbrStreamingCount,
    ServiceTier,
    TotalSpend,
    TotalSpendPct,
    isM2M_dual,
    isM2M_internetOnly,
    isOneYear_dual,
    isTwoYear_dual,
    chargeRatio,
    numServices,
    chargesPerService
FROM derived_variables
;

In [0]:
-- SELECT *
-- FROM workspace.default.train_silver
-- LIMIT 10

DESCRIBE workspace.default.train_silver

## Analysis

### Contract vs Service Mix (Segment Profile)
Decomposes the three contract types by phone-only, internet-only, and dual-service customers.

**Key findings**
1. Dual-service (phone + internet) is 68% of the base and the revenue engine
    - Across all contracts, 1+1 customers dominate: 2,366 + 773 + 699 = 3,838 (68.1%). They carry the highest MonthlyCharges and by far the highest TotalCharges. This is where churn cost is highest, even a modest churn rate here is expensive.

2. The highest-risk cell is now clearly identifiable: M2M + dual-service
    - 2,366 customers (42% of the entire base)
    - Avg MC **$78.35**, highest M2M price, near the two-year dual-service price ($90.43)
    - Avg tenure 19.7 months, short
    - Avg TC only **$1,668** vs. $4,084 (one-year dual) and $5,751 (two-year dual)
    - These customers pay almost as much per month as two-year customers but stay one-third as long. If your overall churn is 26.7%, this cell is very likely 35-45%. This is the single highest-leverage retention target in the dataset.

3. Phone-only is a cheap, short-lived segment, especially on M2M
    - M2M phone-only: $20.38/mo, 9.5-month tenure, $199 TC. Tiny LTV. 419 customers.
    - These are almost certainly the MonthlyCharges decile 1-2 customers from your earlier analysis (churn ~9%). Low churn, low value, not worth retention spend, but also not a churn problem.
    - Interesting: phone-only tenure **rises sharply with contract commitment** (9.5 → 29.0 → 48.2 months). The contract is doing the retention work, not the product.

4. Internet-only is a distinct, mid-value segment
    - M2M internet-only: $37.12/mo, 18.8-month tenure.
    - One-year internet-only: $45.29/mo, 40.3-month tenure.
    - Two-year internet-only: $51.62/mo, 54.5-month tenure, notably long.
    - The two-year internet-only segment has higher median tenure (64) than dual-service two-year (68)? Close, but std dev is 20.6 vs 12.7, so it's more dispersed. Some very long-tenured internet-only customers.

5. The "price premium for flexibility" is now quantified
    - For dual-service customers:
        - M2M:      Avg MC=$78.35 | Avg Tenure=19.7 | Avg TC=$1,668
        - One year: Avg MC=$84.95 | Avg Tenure=46.9 | Avg TC=$4,084
        - Two year: Avg MC=$90.43 | Avg Tenure=62.7 | Avg TC=$5,751
    - **within dual-service**, two-year customers pay the *most* ($90.43).
    - ‼️Within each service tier, longer contracts cost more per month but deliver far more lifetime value. The model must condition on service mix, or `MonthlyCharges` will mislead.

6. Std devs are tight for phone-only, wide for dual-service
    - Phone-only MC std dev: $1.5-$2.5, essentially a flat $20 product.
    - Dual-service MC std dev: $16.7-$18.4, real price dispersion (add-ons, fiber vs DSL).
    - This means MonthlyCharges is uninformative for phone-only and highly informative for dual-service. Consider a service_tier feature and let the model split on it.




In [0]:

SELECT 
    ContractTier,
    hasPhoneService,
    hasInternetService,
    COUNT(*) AS qty,
    AVG(MonthlyCharges),
    MEDIAN(MonthlyCharges),
    STDDEV(MonthlyCharges),
    AVG(TotalCharges),
    MEDIAN(TotalCharges),
    STDDEV(TotalCharges),
    AVG(nbrTenure),
    MEDIAN(nbrTenure),
    STDDEV(nbrTenure)
FROM workspace.default.train_silver
GROUP BY ALL

After creating `ServiceTier` variable and adding the ChurnRate to the table.

**74% of all churn sits in one cell**
- M2M dual-service = 1,119 of 1,504 churned customers = 74.4% of all churn. It's 42% of the base but three-quarters of the losses. Every other cell is secondary. This single cell is the churn problem.

**Key Insights**:
1. The tenure ladder predicted churn almost perfectly
    - Ranking by tenure ascending ≈ ranking by churn descending. 
    - The one exception is M2M internet_only (40.8% churn on 18.8-month tenure), it churns worse than its tenure would suggest. 
    - That's the "no phone anchor" effect: internet-only customers have no bundling lock-in, and M2M removes contractual lock-in. Two lock-ins absent at once.

2. Contract dominates, but service tier modulates it strongly
    - Within each contract, dual-service churns more than phone-only, except M2M internet_only, which is the second-worst cell. The pattern:
        - Dual = highest price → highest churn when uncommitted (47.3% M2M) but lowest when committed (5.2% two-year).
        - Phone-only = low price, low churn everywhere (1.0-17.9%). Cheap and sticky.
        - Internet-only = mid price, high churn when M2M (40.8%), low when committed.
    - The contract is the retention mechanism. Service mix sets the baseline risk. Dual-service is high-stakes: fragile when uncommitted, anchor when committed.

3. The price paradox resolves
    - M2M dual pays $78 and churns at 47%; two-year dual pays $90 and churns at 5%. Higher price with commitment → lower churn. Price is not the driver, commitment is. This kills any "reduce price to retain" strategy for the M2M dual cell; the lever is contract conversion, not discounting.

4. The 26.7% base rate is deeply misleading
    - It's a blend of 1.0% and 47.3% cells. A model optimized on overall accuracy/ROC-AUC will be dominated by the large M2M dual cell anyway (good), but segment-level evaluation is mandatory, a model could look fine overall while misranking the 40.8% internet-only cell or the 15.1% one-year dual upgrade targets.

--- 

**Business Reading**

| Cell | Churn | Frame | Lever |
| --- | --- | --- | --- |
| M2M dual (2,366) | 47.3% | The revenue hemorrhage | Contract conversion to 1yr/2yr; onboarding in first 6-12 mo |
| M2M internet_only (314) | 40.8% | The hidden bleed | Bundle with phone, or convert to contract; small but high-rate |
| One year dual (773) | 15.1% | The renewal cliff | Renewal-window retention; watch spend_deviation (price bump at renewal) |
| M2M phone_only (419) | 17.9% | Low value, don't over-invest | Cheap; let churn |
| Two-year anything | 1-5% | The anchor | Protect; don't disturb |

The M2M dual cell is 42% of customers and 74% of churn, but note it's also $78/mo * 2,366 = $185K/month of recurring revenue at risk. That's the number that justifies the modeling effort.


In [0]:
SELECT 
    ContractTier,
    ServiceTier,
    COUNT(*) AS qty,
    AVG(MonthlyCharges),
    MEDIAN(MonthlyCharges),
    STDDEV(MonthlyCharges),
    AVG(TotalCharges),
    MEDIAN(TotalCharges),
    STDDEV(TotalCharges),
    AVG(nbrTenure),
    MEDIAN(nbrTenure),
    STDDEV(nbrTenure),
    SUM(TargetChurn) AS churned,
    AVG(TargetChurn) AS churn_rate
FROM workspace.default.train_silver
GROUP BY ALL
ORDER BY churn_rate DESC

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT 
    nbrTenure,
    tenure_calc,
    COUNT(*) AS qty,
    AVG(MonthlyCharges),
    MEDIAN(MonthlyCharges),
    STDDEV(MonthlyCharges),
    AVG(TotalCharges),
    MEDIAN(TotalCharges),
    STDDEV(TotalCharges),
    AVG(nbrTenure),
    MEDIAN(nbrTenure),
    STDDEV(nbrTenure),
    SUM(TargetChurn) AS churned,
    AVG(TargetChurn) AS churn_rate
FROM workspace.default.train_silver
GROUP BY ALL
ORDER BY churn_rate DESC

In [0]:
SELECT 
    TargetChurn, 
    MonthlyCharges,
    TotalCharges,
    CustomerFeedback_adj,
    nbrTenure,
    ROUND(TotalCharges / MonthlyCharges, 0) AS AvgTenure,
    (CASE
        -- numeric years -> * 12
        WHEN CustomerFeedback_adj RLIKE '[0-9]+ years?' THEN
        CAST(REGEXP_EXTRACT(CustomerFeedback_adj, '([0-9]+) years?', 1) AS INT) * 12
        -- numeric months
        WHEN CustomerFeedback_adj RLIKE '[0-9]+ months?' THEN
        CAST(REGEXP_EXTRACT(CustomerFeedback_adj, '([0-9]+) months?', 1) AS INT)
        -- spelled-out "one month"
        WHEN lower(CustomerFeedback_adj) RLIKE '\\bone month' THEN 1
        -- spelled-out "one year"
        WHEN lower(CustomerFeedback_adj) RLIKE '\\bone year' THEN 12
        ELSE NULL
    END) AS tenure_months_viaComment,
    COALESCE(
        -- digits: "5 years", "12 months"
        TRY_CAST(REGEXP_EXTRACT(CustomerFeedback_adj, '(?i)([0-9]+)\\s+years?',  1) AS INT) * 12,
        TRY_CAST(REGEXP_EXTRACT(CustomerFeedback_adj, '(?i)([0-9]+)\\s+months?', 1) AS INT),
        -- one word: "five years", "two months"
        CASE lower(REGEXP_EXTRACT(CustomerFeedback_adj, '(?i)\\b([a-z]+)\\s+years?',  1))
            WHEN 'one' THEN 12  WHEN 'two' THEN 24  WHEN 'three' THEN 36
            WHEN 'four' THEN 48 WHEN 'five' THEN 60 WHEN 'six' THEN 72
            WHEN 'seven' THEN 84  WHEN 'eight' THEN 96  WHEN 'nine' THEN 108 END,
        CASE lower(REGEXP_EXTRACT(CustomerFeedback_adj, '(?i)\\b([a-z]+)\\s+months?', 1))
            WHEN 'one' THEN 1   WHEN 'two' THEN 2   WHEN 'three' THEN 3
            WHEN 'four' THEN 4  WHEN 'five' THEN 5  WHEN 'six' THEN 6 
            WHEN 'seven' THEN 7  WHEN 'eight' THEN 8  WHEN 'nine' THEN 9 END
    ) AS tenure_months_viaComment2
FROM workspace.default.train_silver
WHERE nbrTenure is NULL
-- limit 10




In [0]:
SELECT CustomerFeedback
FROM workspace.default.inference

## Regression to impute TotalCharges

Variables not to use:
- CustomerFeedback_adj

Observations:
- 1 `tenure` that is NULL when `TotalCharges` is NULL
- Inference needs to be done for tenure can be based on MonthlyCharges?


In [0]:
-- Checking TotalCharges missing
SELECT
    nbrTenure,
    tenure_calc,
    TotalCharges,
    MonthlyCharges
FROM workspace.default.train_silver
WHERE TotalCharges IS NULL

In [0]:

WITH
charge_band AS (
    SELECT
        ContractTier,
        ChargeBand,
        ServiceTier,
        ROUND(MEDIAN(nbrTenure), 0) AS cohort_median_tenure,
        ROUND(AVG(nbrTenure), 0) AS cohort_mean_tenure,
        STDDEV(nbrTenure) AS cohort_std_tenure
    FROM workspace.default.train_silver
    GROUP BY ALL
)

SELECT
    t1.CustomerID,
    t1.nbrTenure,
    t1.tenure_calc,
    t2.cohort_median_tenure,
    t2.cohort_mean_tenure,
    t2.cohort_std_tenure,
    TRY_ADD(t1.nbrTenure, -1*t2.cohort_mean_tenure) AS diff1,
    TRY_ADD(t1.tenure_calc, -1*t2.cohort_mean_tenure) AS diff2

FROM
    workspace.default.train_silver AS t1
    INNER JOIN charge_band AS t2
        ON t1.ContractTier = t2.ContractTier
        AND t1.ChargeBand = t2.ChargeBand
        AND t1.ServiceTier = t2.ServiceTier
WHERE
    t1.nbrTenure IS NULL


In [0]:
SELECT * 
FROM workspace.default.train_silver
WHERE CustomerID = '9651-GTSAQ'

In [0]:
SELECT AVG(nbrTenure)
FROM workspace.default.train_silver